# Palace Width Sweep: Parallel Simulation

Sweep the microstrip width from 5 to 50 um (5 points), run all simulations
in parallel on GDSFactory+ cloud using the non-blocking API, then compare
S11 and S21 across widths.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

### Define the sweep

In [1]:
import numpy as np

widths = np.arange(2, 21, 4)

### Build components and configure simulations

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK, cells

from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

PDK.activate()

stack = get_stack(air_above=300.0)  # auto-detects active PDK

sims = []

for w in widths:
    # Build component
    c = gf.Component()
    r1 = c << cells.straight_metal(length=1000, width=w)

    r = c.get_region(layer=LAYER.TopMetal2drawing)
    r_sized = r.sized(+20000)
    c.add_polygon(r_sized, layer=LAYER.Metal1drawing)
    c.add_ports(r1.ports)

    # Configure simulation
    sim = DrivenSim()
    sim.set_output_dir(f"./palace-sim-w{w:.1f}")
    sim.set_geometry(c)
    sim.set_stack(stack)

    for port in c.ports:
        sim.add_port(
            port.name, from_layer="metal1", to_layer="topmetal2", geometry="via"
        )

    sim.set_driven(fmin=1e9, fmax=100e9, num_points=80)
    sim.mesh(preset="default")

    sims.append(sim)

print(f"Configured {len(sims)} simulations")

### Upload and start all jobs (non-blocking)

In [3]:
# Upload and start all jobs without waiting
job_ids = []
for sim in sims:
    job_id = sim.run(wait=False)
    job_ids.append(job_id)

print(f"Started {len(job_ids)} jobs: {job_ids}")

Job started: palace-e685826f


Job started: palace-86abae3d


Job started: palace-09114741


Job started: palace-99ccfb9e


Job started: palace-6a007652
Started 5 jobs: ['019d5483-5e24-7640-8e85-2548b29bd2f2', '019d5483-6b66-7a70-8624-3fb570ec12f5', '019d5483-7885-7d52-a95c-cd7394c03063', '019d5483-857a-7791-9638-0647d17f78b3', '019d5483-9279-7580-97fe-cd9ec3859aee']


### Wait for all jobs to complete

In [4]:
import gsim

# Poll all jobs concurrently, download and parse results
results = gsim.wait_for_results(job_ids)

Waiting for 5 jobs...
  palace-e685826f                completed    2m 33s
  palace-86abae3d                completed    2m 59s
  palace-09114741                completed    4m 31s
  palace-99ccfb9e                completed    2m 45s
  palace-6a007652                completed    3m 21s


Extracting results.tar.gz...
Downloaded 10 files to sim-data-palace-e685826f


Extracting results.tar.gz...
Downloaded 10 files to sim-data-palace-86abae3d


Extracting results.tar.gz...
Downloaded 10 files to sim-data-palace-09114741


Extracting results.tar.gz...
Downloaded 10 files to sim-data-palace-99ccfb9e


Extracting results.tar.gz...
Downloaded 10 files to sim-data-palace-6a007652


### Plot S11 and S21 comparison

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6))

for w, sp in zip(widths, results, strict=True):
    ax1.plot(sp.freq, sp.s11.db, label=f"w={w:.1f} um")
    ax2.plot(sp.freq, sp.s21.db, label=f"w={w:.1f} um")

ax1.set(
    xlabel="Frequency (GHz)", ylabel="|S11| (dB)", title="S11 — Return Loss vs Width"
)
ax1.legend()
ax1.grid(True)

ax2.set(
    xlabel="Frequency (GHz)", ylabel="|S21| (dB)", title="S21 — Insertion Loss vs Width"
)
ax2.legend()
ax2.grid(True)

plt.tight_layout()